In [0]:
import math
from collections import Counter

def analyze_characters(text_string, target_chars):
    # Convert list of target characters to a set for fast O(1) lookups
    target_set = set(target_chars)

    text_string = " ".join(text_string)
    
    # 1. Split the string into individual lines
    lines = text_string.strip().split('\n')
    num_lines = len(lines)
    
    if num_lines == 0:
        return "The provided string is empty."

    # Tracking frequency of target characters per line
    # Structure: {char: [count_in_line1, count_in_line2, ...]}
    char_counts_per_line = {char: [0] * num_lines for char in target_set}
    
    # 2. Count occurrences across all lines
    for line_idx, line in enumerate(lines):
        line_counts = Counter(line)
        for char in target_set:
            char_counts_per_line[char][line_idx] = line_counts[char]

    # Metrics storage
    results = {}
    
    # 3. Calculate statistics for each character
    for char, counts in char_counts_per_line.items():
        total_occurrence = sum(counts)
        mean = total_occurrence / num_lines
        
        # Calculate Variance and Standard Deviation for consistency
        variance = sum((x - mean) ** 2 for x in counts) / num_lines
        std_deviation = math.sqrt(variance)
        
        results[char] = {
            "total_occurrence": total_occurrence,
            "consistency_score_std": round(std_deviation, 2),
            "counts_per_line": counts
        }
        
    # Find the most occurring and most consistent characters
    # Lower standard deviation means MORE consistent
    most_occurring = max(results.keys(), key=lambda c: results[c]["total_occurrence"])
    most_consistent = min(results.keys(), key=lambda c: results[c]["consistency_score_std"])
    
    return results, most_occurring, most_consistent

# --- Example Usage ---
multiline_str = [
    ";;;;;;",
    "profile_id;name;experience;;;;",
    "1;John Doe;Built ETL pipelines;;;;",
    "2;Alice Smith;SQL dashboards;;;;",
    "3;Bob Lee;ML models;;;;"
]
search_list = [",", ";", "|", "\t"]

stats, highest, consistent = analyze_characters(multiline_str, search_list)

print(f"Analysis Results: {stats}\n")
print(f"Character that occurs the MOST: '{highest}'")
print(f"Character that is MORE CONSISTENT: '{consistent}'")


In [0]:
def detect_has_header(text_string, target_chars=[",", ";", "|", "\t"]):
    lines = [line.strip() for line in text_string.strip().split('\n') if line.strip()]
    if len(lines) < 3:
        return False
        
    # Count how often target characters appear in the first row vs the rest
    first_row_counts = [lines[0].count(c) for c in target_chars]
    
    # Calculate average counts for the data rows (rows 1 to end)
    data_rows_counts = []
    for c in target_chars:
        avg_in_data = sum(line.count(c) for line in lines[1:]) / (len(lines) - 1)
        data_rows_counts.append(avg_in_data)
        
    # If the character density in row 1 differs heavily from the data average,
    # it implies a structural difference (e.g., text headers vs numeric data)
    variance_detected = any(abs(f - d) > 1.5 for f, d in zip(first_row_counts, data_rows_counts))
    
    return variance_detected  # Returns True/False Boolean


In [0]:
def detect_delimiter(text_string, candidate_delimiters=[',', ';', '\t', '|']):
    text_string = "\n".join(text_string)
    print(text_string)
    lines = [line.strip() for line in text_string.strip().split('\n') if line.strip()]
    num_lines = len(lines)
    
    if num_lines < 1:
        return ","  # Default fallback if not enough data
        
    for char in candidate_delimiters:
        # Count occurrences of this candidate in the first line
        expected_count = lines[0].count(char)
        
        # A valid delimiter must actually exist in the row
        if expected_count == 0:
            continue
            
        # Check if EVERY other line has the exact same count
        if all(line.count(char) == expected_count for line in lines):
            return char  # Return immediate string format

    return None  # No clear delimiter found


detect_delimiter([
    ";;;;;;",
    "profile_id;name;experience;;;;",
    "1;John Doe;Built ETL pipelines;;;;",
    "2;Alice Smith;SQL dashboards;;;;",
    "3;Bob Lee;ML models;;;;"
], candidate_delimiters=[',', ';', '\t', '|'])


In [0]:
def detect_delimiter(lines_list, candidate_delimiters=[',', ';', '\t', '|']):
    # Quick guard: handle empty inputs or single-line cases immediately
    if not lines_list:
        return ","
    if len(lines_list) == 1:
        # If only 1 line, pick the candidate that appears the most
        first_line = lines_list[0]
        return max(candidate_delimiters, key=first_line.count, default=",")

    # Fast validation: check candidates one by one
    for char in candidate_delimiters:
        # Get the target count from the first row using C-optimized count()
        expected_count = lines_list[0].count(char)
        
        # Delimiters must exist (count > 0)
        if expected_count == 0:
            continue
            
        # Short-circuit evaluation: 'all()' stops the instant a line mismatch is found.
        # This avoids processing the rest of the dataset.
        if all(line.count(char) == expected_count for line in lines_list):
            return char  

    # Fallback: if no perfect match, return the most frequent candidate in row 1
    return max(candidate_delimiters, key=lines_list[0].count, default=",")

# --- Test Data ---
data_lines = [
    ";;;;;;",
    "profile_id;name;experience;;;;",
    "1;John Doe;Built ETL pipelines;;;;",
    "2;Alice Smith;SQL dashboards;;;;",
    "3;Bob Lee;ML models;;;;"
]

print(f"Detected Delimiter: '{detect_delimiter(data_lines)}'")
# Output: Detected Delimiter: ';'



In [0]:
def detect_delimiter(text_input, candidate_delimiters=[',', ';', '\t', '|']):
    # 1. Normalization Step: Safely convert any input format into a list of strings
    if isinstance(text_input, str):
        # Handles raw multi-line strings safely
        lines = [line.strip() for line in text_input.splitlines() if line.strip()]
    elif isinstance(text_input, (list, tuple, set)):
        # Handles lists of strings, making sure to filter out empty rows
        lines = [str(line).strip() for line in text_input if str(line).strip()]
    else:
        return ","  # Fallback for unexpected data types
        
    # Guard check for empty data
    if not lines:
        return ","
        
    # 2. Vectorized-style counting with native Python short-circuiting
    for char in candidate_delimiters:
        # Check the first row's count at bare-metal C speed
        expected_count = lines[0].count(char)
        
        # Valid delimiters must exist in the row
        if expected_count == 0:
            continue
            
        # Short-circuit logic: the moment ONE line doesn't match, 
        # Python immediately abandons this character and checks the next candidate.
        if all(line.count(char) == expected_count for line in lines):
            return char

    # 3. Fallback: If no delimiter is perfectly consistent, 
    # return the most frequent candidate found in the first row
    return max(candidate_delimiters, key=lines[0].count, default=",")

# --- Verification ---
data_lines = [
    ";;;;;;",
    "profile_id;name;experience;;;;",
    "1;John Doe;Built ETL pipelines;;;;",
    "2;Alice Smith;SQL dashboards;;;;",
    "3;Bob Lee;ML models;;;;"
]

print(f"Detected Delimiter: '{detect_delimiter(text_input)}'")
detect_delimiter(text_input)
# Output: Detected Delimiter: ';'


In [0]:
def detect_delimiter(text_input, candidate_delimiters=[',', ';', '\t', '|']):
    """
    Detect the most likely delimiter in raw text input.

    Returns:
    {
        "status": "SUCCESS" | "FAILED",
        "result": {
            "delimiter": str | None
        },
        "metadata": {
            "confidence": float,
            "rows_analyzed": int,
            "candidate_scores": dict
        }
    }
    """

    import builtins

    # -----------------------------
    # 1. Normalize input
    # -----------------------------
    if isinstance(text_input, str):
        lines = [l.strip() for l in text_input.splitlines() if l.strip()]
    elif isinstance(text_input, (list, tuple, set)):
        lines = [str(l).strip() for l in text_input if str(l).strip()]
    else:
        return {
            "status": "FAILED",
            "result": {"delimiter": None},
            "metadata": {
                "confidence": 0.0,
                "rows_analyzed": 0,
                "candidate_scores": {}
            }
        }

    if not lines:
        return {
            "status": "FAILED",
            "result": {"delimiter": None},
            "metadata": {
                "confidence": 0.0,
                "rows_analyzed": 0,
                "candidate_scores": {}
            }
        }

    # -----------------------------
    # 2. Scoring candidates
    # -----------------------------
    candidate_scores = {}
    best_delimiter = None
    best_score = -1

    for d in candidate_delimiters:
        column_counts = []

        for line in lines:
            column_counts.append(len(line.split(d)))

        if not column_counts:
            continue

        most_common = builtins.max(
            set(column_counts),
            key=column_counts.count
        )

        stability = column_counts.count(most_common)
        avg_cols = sum(column_counts) / len(column_counts)

        # penalize meaningless splits
        if avg_cols < 2:
            score = 0
        else:
            score = stability * avg_cols

        candidate_scores[d] = score

        if score > best_score:
            best_score = score
            best_delimiter = d

    # -----------------------------
    # 3. Confidence calculation
    # -----------------------------
    total = sum(candidate_scores.values()) or 1
    confidence = probabilities = score / sum(candidate_scores)

    # -----------------------------
    # 4. Return standard format
    # -----------------------------
    return {
        "status": "SUCCESS" if best_delimiter else "FAILED",
        "result": {
            "delimiter": best_delimiter
        },
        "metadata": {
            "confidence": round(confidence, 4),
            "rows_analyzed": len(lines),
            "candidate_scores": candidate_scores
        }
    }
text_input = [
    ";;;;;;",
    "profile_id;name;experience;;;;",
    "1;John Doe;Built ETL pipelines;;;;",
    "2;Alice Smith;SQL dashboards;;;;",
    "3;Bob Lee;ML models;;;;"
]
detect_delimiter(text_input)

In [0]:
# now we will create a function to detect headers 
def detect_header(text_input, detect_delimiter_output):
    pass




In [0]:

def detect_header(lines, delimiter):
    """
    Detects header row in messy raw data.
    """

    # -----------------------------
    # 1. Normalize
    # -----------------------------
    cleaned = [l.strip() for l in lines if l and l.strip()]

    if not cleaned:
        return {
            "status": "FAILED",
            "result": {"header_row_index": None},
            "metadata": {"confidence": 0.0}
        }

    # -----------------------------
    # 2. Score each row
    # -----------------------------
    scores = []

    for i, line in enumerate(cleaned):
        parts = line.split(delimiter)

        # feature 1: number of columns
        col_count = len(parts)

        # feature 2: text vs numeric ratio
        alpha_ratio = sum(p.replace("_", "").isalpha() for p in parts) / max(col_count, 1)

        # feature 3: similarity with next row (structure check)
        if i + 1 < len(cleaned):
            next_parts = cleaned[i + 1].split(delimiter)
            structure_match = 1 if len(next_parts) == col_count else 0
        else:
            structure_match = 0

        # final score (weighted heuristic)
        score = (alpha_ratio * 0.6) + (structure_match * 0.4)

        scores.append(score)

    # -----------------------------
    # 3. Pick best candidate
    # -----------------------------
    best_index = max(range(len(scores)), key=lambda i: scores[i])
    best_score = scores[best_index]

    # -----------------------------
    # 4. Extract header columns
    # -----------------------------
    header_columns = cleaned[best_index].split(delimiter)

    # -----------------------------
    # 5. Return structured output
    # -----------------------------
    return {
        "status": "SUCCESS",
        "result": {
            "header_row_index": best_index,
            "header_columns": header_columns
        },
        "metadata": {
            "confidence": round(best_score, 4),
            "rows_scored": len(scores)
        }
    }
delimiter = detect_delimiter(lines)

lines = [
    "Generated by HR System",
    "Export Date: 2026-06-20",
    "Department: Data Analytics",
    "",
    "profile_id;name;experience",
    "1;John Doe;Built ETL pipelines with Spark and Airflow",
    "2;Alice Smith;Worked on SQL dashboards and Power BI",
    "3;Bob Lee;Developed ML models using Python and Pandas",
    "4;Sara Khan;Built data lake pipelines in Databricks"
]

detect_header(lines, delimiter["result"]["delimiter"]) 